# Lecture : Integrating GNNs and LLMs

## Lab 01 : TAPE for boosting GNN predictions with LLM features -- Solution

### Xavier Bresson, Zhou Wei

X. He, X. Bresson, T. Laurent, A. Perold, Y. LeCun, B. Hooi, Harnessing Explanations: LLM-to-LM Interpreter for Enhanced Text-Attributed Graph Representation Learning, 2024, https://arxiv.org/pdf/2305.19523



In [1]:
# For Google Colaboratory
# Go to Runtime → Change runtime type → Select T4 GPU to accelerate LM embedding computation
import sys, os
if "google.colab" in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5284_2026_codes/08_LLMs_GNNs'
    print(path_to_file)
    # change current path to the folder containing "path_to_file"
    os.chdir(path_to_file)
    !pwd
    !pip -q install -f https://data.dgl.ai/wheels/torch-2.6/repo.html dgl==2.5.0 # Install DGL


## GNN + LLM text embeddings for Cora Classification

In TAPE, large language model information is used to enrich node representations before graph learning.

This lab studies a simpler version of TAPE with the pipeline :

```text
Cora dataset
        |
        +--> standard BoW node features --> GCN
        |
        +--> title + abstract --> pre-trained LM --> enhanced node features --> GCN
```


## Setup

The next cell installs only the packages needed for the language-model part of this notebook: `transformers`, `sentencepiece`,  `safetensors`. 

The existing PyTorch/DGL environment is left unchanged. 

The language model is `HuggingFaceTB/SmolLM2-360M`.



In [2]:
# Packages needed for the LM part of this notebook
import importlib.util
import subprocess
import sys
required = {}
required["transformers"] = "transformers==4.46.3"
required["sentencepiece"] = "sentencepiece==0.2.0"
required["safetensors"] = "safetensors>=0.4.5"
required["truststore"] = "truststore==0.10.5"
missing = []
for module_name, pip_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        missing.append(pip_name)
if missing:
    print("Installing:", " ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("LM dependencies already installed")
    

LM dependencies already installed


## Imports


In [3]:
# We disable Hugging Face's Xet transfer backend as it can cause downloading issue
# Xet must be disable before importing `transformers`
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
# Use the OS trust store for HTTPS certificate verification.
import truststore
truststore.inject_into_ssl()
import random
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
from dgl.nn import GraphConv
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
print("PyTorch:", torch.__version__)
print("DGL:", dgl.__version__)


PyTorch: 2.1.2
DGL: 1.0.2


## Reproducibility and compute devices

The next cell selects the random seed for Python, NumP, and PyTorch for reproducibility.

Then it chooses devices separately for the LM and the GCN. 


In [4]:
# Reproducibility and devices
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use CUDA for LM embeddings if a GPU is available; otherwise use CPU.
lm_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
# DGL in the course environment supports CPU, but not Apple's MPS backend.
# Therefore the GCN runs on CPU.
gnn_device = torch.device("cpu")
print("LM device :", lm_device)
print("GCN device:", gnn_device)


LM device : cpu
GCN device: cpu


## Load the Cora dataset

The course dataset is provided in

```text
dataset/cora/cora_prepared.pt
```

If the file is missing, run `code01_dataset.ipynb` to generate it.

The dataset contains :

- the citation DGL graph,
- the standard Cora bag-of-words node features,
- the class labels,
- the train / validation / test masks,
- the title and abstract associated with each graph node.

The node-feature representations used in this notebook are

$$
X_{\text{BoW}}\in\mathbb{R}^{n\times 1433}
\qquad\text{and}\qquad
X_{\text{LM}}\in\mathbb{R}^{n\times d_{LM}},
$$

where $n$ is the number of graph nodes and $d_{LM}$ is the hidden-state dimension of the selected language model.




In [5]:
# Load the instructor-prepared Cora dataset from disk
DATA_DIR = Path("./dataset/cora")
PREPARED_DATA_PATH = DATA_DIR / "cora_prepared.pt"
if not PREPARED_DATA_PATH.exists():
    raise FileNotFoundError( f"Missing course dataset: {PREPARED_DATA_PATH}\n"
                             "Place the instructor-provided prepared dataset in the dataset/ folder.")
try:
    data = torch.load(PREPARED_DATA_PATH, map_location="cpu", weights_only=False)
except TypeError:
    data = torch.load(PREPARED_DATA_PATH, map_location="cpu")

# Reconstruct the citation graph and load the standard node features and labels
src = data["src"].long()
dst = data["dst"].long()
num_nodes = int(data["num_nodes"])
g = dgl.graph((src, dst), num_nodes=num_nodes)
x_standard = data["x_standard"].float()
y = data["y"].long()

# Load the boolean masks that define the training, validation and test node splits
train_mask = data["train_mask"].bool()
val_mask = data["val_mask"].bool()
test_mask = data["test_mask"].bool()

# Load the aligned paper texts, paper IDs, and class names used throughout the experiment
texts = list(data["texts"])
paper_ids = list(data["paper_ids"])
class_names = list(data["class_names"])
num_classes = len(class_names)

print(g)
print("standard features:", tuple(x_standard.shape))
print("classes:", num_classes)
print("aligned texts:", len(texts))
print("train/val/test:", train_mask.sum().item(), val_mask.sum().item(), test_mask.sum().item())

# Print one randomly selected article title and abstract
random_idx = random.randrange(len(texts))
print(f"\nRandom article (node {random_idx}, paper ID {paper_ids[random_idx]}):\n")
print(texts[random_idx])


Graph(num_nodes=2708, num_edges=13264,
      ndata_schemes={}
      edata_schemes={})
standard features: (2708, 1433)
classes: 7
aligned texts: 2708
train/val/test: 270 1083 1355

Random article (node 1577, paper ID 82087):

Title: Fast Probabilistic Modeling for Combinatorial Optimization
Abstract: Probabilistic models have recently been utilized for the optimization of large combinatorial search problems. However, complex probabilistic models that attempt to capture inter-parameter dependencies can have prohibitive computational costs. The algorithm presented in this paper, termed COMIT, provides a method for using probabilistic models in conjunction with fast search techniques. We show how COMIT can be used with two very different fast search algorithms: hillclimbing and Population-based incremental learning (PBIL). The resulting algorithms maintain many of the benefits of probabilistic modeling, with far less computational expense. Extensive empirical results are provided; COMIT ha

## Turn text into vectors with a small pre-trained language model : SmolLM2-360M

We use the hidden states of a pre-trained decoder-only transformer as an encoder and mean-pool only non-padding tokens.

The model is :

```python
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
```

Its weights are about 720 MB.

The idea is to embed the 2,708 paper as vectors :

```text
title + abstract -> pre-trained language model -> dense node vector 
```


## Configure the text encoder

We choose the pre-trained LM, token limit, batch size and cache filename. 

Each text is at most MAX_LENGTH=256 tokens, and embeddings are computed in batches of 8. 

The cache filename includes the model name and token limit.


In [6]:
# LM model: 360M parameters, 720 MB weights
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
MAX_LENGTH = 256
BATCH_SIZE = 8
safe_model_name = MODEL_NAME.replace("/", "__")
cache_path = DATA_DIR / f"cora_{safe_model_name}_meanpool_L{MAX_LENGTH}.pt"
cache_path.parent.mkdir(exist_ok=True)
print("Model:", MODEL_NAME)
print("cache:", cache_path)


Model: HuggingFaceTB/SmolLM2-360M
cache: dataset/cora/cora_HuggingFaceTB__SmolLM2-360M_meanpool_L256.pt


## Convert text into one vector per node

Next, we tokenize titles and abstracts, pass them through the pre-trained transformer, mean-pool the final hidden states over non-padding tokens, and then $L_2$-normalize each paper vector. 

Gradients are disabled with `@torch.no_grad()` because the LM is used only as a fixed feature extractor.

If $\mathbf{h}_{it}$ is the final hidden state of token $t$ for paper $i$, and $m_{it}\in\{0,1\}$ is its padding mask, masked mean pooling is defined as 

$$
\mathbf{s}_i
=
\frac{\sum_t m_{it}\mathbf{h}_{it}}
{\max\!\left(1,\sum_t m_{it}\right)}\in\mathbb{R}^{d_{LM}},
$$

where $d_{LM}=960$ for SmolLM2-360M.

The pooled vector is then normalized :

$$
\mathbf{z}_i
=
\frac{\mathbf{s}_i}
{\|\mathbf{s}_i\|_2}\in\mathbb{R}^{d_{LM}}.
$$

The resulting $\mathbf{z}_i$ becomes the language-model node feature for paper $i$.

### Exercise 1: Masked mean pooling
Complete the two marked operations: sum only non-padding token vectors and count valid tokens.


In [7]:
# Mean-pool token embeddings while ignoring padding positions using the attention mask
def masked_mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    # YOUR CODE STARTS
    summed = ...  # sum masked token vectors over the token dimension
    count = ...  # count non-padding tokens per text; clamp the minimum to 1
    # YOUR CODE ENDS
    return summed / count

# Encode all paper texts in batches with a pretrained language model and return normalized embeddings
@torch.no_grad()
def embed_texts(texts, model_name, batch_size=8, max_length=256, device=lm_device):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Use float16 on CUDA to save GPU memory, and float32 on CPU.
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    model = AutoModel.from_pretrained(model_name, torch_dtype=dtype).to(device)
    model.eval()
    out = []
    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]
        batch = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        h = model(**batch).last_hidden_state
        z = masked_mean_pool(h, batch["attention_mask"])
        z = F.normalize(z.float(), p=2, dim=1)
        out.append(z.cpu())
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return torch.cat(out, dim=0)


For the first run, the code automatically downloads the model weights from Hugging Face, compute the 2,708 embeddings, cashed the resulting PyTorch tensors in 

```text
dataset/cora_HuggingFaceTB__SmolLM2-360M_meanpool_L256.pt
```

Thus, if an embedding tensor is already cached, it is loaded immediately for the next runs. 

For the experiment, the feature matrix has one row per graph node $X_{\text{LM}}\in\mathbb{R}^{2708\times 960}$.


In [ ]:
# Load cached language model embeddings when available, otherwise compute and save them
if cache_path.exists():
    x_lm = torch.load(cache_path, map_location="cpu")
    print("Loaded cached embeddings")
else:
    texts_to_embed = texts
    x_lm = embed_texts(texts_to_embed, MODEL_NAME, batch_size=BATCH_SIZE, max_length=MAX_LENGTH).cpu()
    torch.save(x_lm, cache_path)
    print("Saved embeddings to:", cache_path)
print("LM features :", tuple(x_lm.shape))


  0%|          | 0/339 [00:00<?, ?it/s]

## Model 1 : A Baseline 2-layer GCN with standard Cora features

The architecture is a two-layer graph convolutional network :
- The first graph convolution maps node features to a hidden representation,
- A ReLU layer is applied,
- A dropout layer regularizes the hidden representation,
- The second and final graph convolution produces one logit per class. 

Ignoring notation for bias terms, a DGL `GraphConv` with its default symmetric normalization can be written as

$$
\hat A=D^{-1/2}AD^{-1/2}\in\mathbb{R}^{n\times n},\ \textrm{ with } n=2708,
$$

where the prepared graph $A$ already contains self-loops. The two layers are approximately

$$
H^{(1)}=\operatorname{ReLU}(\hat AXW_0+b_0)\in\mathbb{R}^{n\times d_{GCN} }, 
$$

$$
Z=\hat A\,\operatorname{Dropout}(H^{(1)})W_1+b_1\in\mathbb{R}^{n\times d_{GCN} },
$$

with

$$
X\in\mathbb{R}^{n\times d_{\text{input}} }, \ \textrm{ with } d_{\text{input}}=1433.
$$

The predicted class for node $i$ is

$$
\hat y_i=\operatorname*{arg\,max}_{c\in\{0,\ldots,n_c-1\}} Z_{ic}\in\{0,\ldots,n_c-1\},
$$

and accuracy is the fraction of nodes for which $\hat y_i=y_i$.

We will use $d_{GCN}=64$.

### Exercise 2: GCN forward pass
Complete the two-layer forward pass. The first GraphConv already applies ReLU; use training-aware dropout between the layers.


In [ ]:
# Define a two-layer graph convolutional network with ReLU activation and dropout
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GraphConv(in_dim, hidden_dim, activation=F.relu)
        self.conv2 = GraphConv(hidden_dim, out_dim)
        self.dropout = dropout
    def forward(self, graph, x):
        # YOUR CODE STARTS
        x = ...  # first graph convolution
        x = ...  # dropout, active only during training
        x = ...  # final graph convolution
        # YOUR CODE ENDS
        return x

# Compute classification accuracy from the highest-scoring predicted class for each node
def accuracy(logits, labels):
    predictions = logits.argmax(dim=1)
    correct = predictions == labels
    return correct.float().mean().item()


## Train with validation-based early stopping

The GCN is trained using the row-normalized 1,433-dimensional Cora bag-of-words matrix as node features. This run can be summarized as

$$
Z_{\text{BoW}}
=
\operatorname{GCN}\!\left(G,X_{\text{BoW}}\right)\in\mathbb{R}^{n\times n_c}.
$$

It provides the baseline against which the language model features will be compared.

We monitor the validation accuracy and keep the best validation checkpoint, and stop after `patience` epochs without improvement. 

We also use a test mask that prevents the test set from being used during training.


In [ ]:
# Train a GCN with early stopping on validation accuracy and evaluate the best model on the test set
def train_gcn(graph, features, labels, train_mask, val_mask, test_mask, 
              hidden_dim=64, lr=0.01, weight_decay=0.0, epochs=300, patience=50, seed=0):

    # Fix the PyTorch random seed so model initialization is reproducible
    torch.manual_seed(seed)

    # Move the graph, features, labels, and split masks to the selected GNN device
    graph = graph.to(gnn_device)
    features = features.to(gnn_device)
    labels = labels.to(gnn_device)
    train_mask = train_mask.to(gnn_device)
    val_mask = val_mask.to(gnn_device)
    test_mask = test_mask.to(gnn_device)
    output_dim = int(labels.max().item()) + 1

    # Create the GCN using the input feature dimension and the number of target classes
    model = GCN(in_dim=features.shape[1], hidden_dim=hidden_dim, out_dim=output_dim).to(gnn_device)

    # Create an Adam optimizer for updating the GCN parameters during training
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Initialize variables used to track the best validation checkpoint and training history
    best_val = -1.0
    best_state = None
    wait = 0
    history = []
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        logits = model(graph, features)
        loss = F.cross_entropy(logits[train_mask], labels[train_mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            logits = model(graph, features)
            train_acc = accuracy(logits[train_mask], labels[train_mask])
            val_acc = accuracy(logits[val_mask], labels[val_mask])
        history.append((epoch, loss.item(), train_acc, val_acc))
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    # Restore the best validation checkpoint and evaluate its accuracy on the test nodes
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(graph, features)
        test_acc = accuracy(logits[test_mask], labels[test_mask])

    elapsed = time.time() - t0
    history_df = pd.DataFrame(history, columns=["epoch", "loss", "train_acc", "val_acc"])

    result = {}
    result["model"] = model.cpu()
    result["history"] = history_df
    result["best_val_acc"] = best_val
    result["test_acc"] = test_acc
    result["train_seconds"] = elapsed
    return result


# Train and evaluate the baseline GCN using the standard Cora bag-of-words node features
result_standard = train_gcn(g, x_standard, y, train_mask, val_mask, test_mask, 
                           hidden_dim=64, lr=0.01, weight_decay=0.0, epochs=300, patience=50, seed=SEED)
print(f"best validation accuracy = {result_standard['best_val_acc']:.4f}")
print(f"test accuracy            = {result_standard['test_acc']:.4f}")
print(f"GCN training time        = {result_standard['train_seconds']:.1f} s")


## Model 2 : GCN with title + abstract LM-embeddings 

We repeat the same GCN training procedure, but replace the standard bag-of-words features with the pre-trained language-model embeddings.

The only intended experimental change is the input feature matrix:

$$
Z_{\text{LM}}
=
\operatorname{GCN}\!\left(G,X_{\text{LM}}\right)\in\mathbb{R}^{n\times n_c}.
$$

### Exercise 3: Use LM features
Run the same GCN with the text embeddings from the previous section. Keep the graph, labels, masks, and training settings unchanged.


In [ ]:
# Train and evaluate the same GCN using the language-model node embeddings
# YOUR CODE STARTS
lm_features = ...  # use the embeddings computed in the text-encoder section
# YOUR CODE ENDS
result_lm = train_gcn(g, lm_features, y, train_mask, val_mask, test_mask, 
                           hidden_dim=64, lr=0.01, weight_decay=0.0, epochs=300, patience=50, seed=SEED)
print(f"best validation accuracy = {result_lm['best_val_acc']:.4f}")
print(f"test accuracy            = {result_lm['test_acc']:.4f}")
print(f"GCN training time        = {result_lm['train_seconds']:.1f} s")


## Compare standard GCN and GCN augmented with LM features


In [ ]:
standard_row = {}
standard_row["features"] = "Model 1: GCN on Cora standard BoW"
standard_row["dim"] = x_standard.shape[1]
standard_row["best_val_acc"] = result_standard["best_val_acc"]
standard_row["test_acc"] = result_standard["test_acc"]
standard_row["gcn_train_sec"] = result_standard["train_seconds"]

lm_row = {}
lm_row["features"] = f"Model 2: SmolLM2-360M on title+abstract"
lm_row["dim"] = x_lm.shape[1]
lm_row["best_val_acc"] = result_lm["best_val_acc"]
lm_row["test_acc"] = result_lm["test_acc"]
lm_row["gcn_train_sec"] = result_lm["train_seconds"]

results = pd.DataFrame([standard_row, lm_row])
results
